In [ ]:
!pip install transformers seqeval evaluate accelerate -U
!pip install transformers seqeval evaluate accelerate pytorch-crf -U

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 19.9 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=a73cc2fbec2969f40177971c16a7a4079d79b4be2af2d5e4181f24f9c160708b
  Stored in directory: /root/.cache/pip/wheels/14/cf/a7/8f28ef376d707ff10e3922899482a2f23ef3002f8a952f47ac
Successfully built seqeval
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.15.1
    Uninstalling transformers-5.15.1:
      Successfully uninstalled transformers-5.15.1


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
label_path = '/content/drive/MyDrive/datasetViMedNER/traindata/labels.txt'

unique_tags = []
with open(label_path, "r", encoding = "utf-8") as f :
    for line in f:
        line.strip()
        if line.strip():
          unique_tags.append(line.strip())

label2id = {tag: idx for idx, tag in enumerate(unique_tags)}
id2label = {idx: tag for idx, tag in enumerate(unique_tags)}

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel
from transformers.modeling_outputs import TokenClassifierOutput
from torchcrf import CRF

class PhoBERT_CRF(nn.Module):
    def __init__(self, model_checkpoint, num_labels):
        super().__init__()
        self.phobert = AutoModel.from_pretrained(model_checkpoint, add_pooling_layer = False)
        self.classifier = nn.Linear(self.phobert.config.hidden_size, num_labels)
        self.crf = CRF(num_labels, batch_first = True)

    def forward(self, input_ids, attention_mask, labels = None, **kwargs):
        outputs = self.phobert(input_ids = input_ids, attention_mask = attention_mask) # đưa qua phobert lấy ngữ cảnh câu

        sequence_output = outputs.last_hidden_state

        emissions = self.classifier(sequence_output) # đưa ra kết quả thô của từng nhãn

        loss = None
        if labels is not None:
            crf_mask = attention_mask.bool() # chuyển nhãn không hợp lệ thành True False để xác định những vị trí cần tính crf

            # đổi quy ước của labels khi đưa qua crf
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0 #các labels padding, sub_token chuyển nhãn 0

            loss = -self.crf(emissions, safe_labels, mask = crf_mask, reduction = 'mean')

        # tạo một ma trận kết quả phụ để mô hình có thể sử dụng kết quả
        crf_mask_decode = attention_mask.bool()
        decoded_paths = self.crf.decode(emissions, mask = crf_mask_decode)

        fake_logits = torch.zeros_like(emissions)
        for i, path in enumerate(decoded_paths):
            for j, tag_id in enumerate(path):
                fake_logits[i, j, tag_id] = 1.0

        return TokenClassifierOutput(loss=loss, logits=fake_logits)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel
from transformers.modeling_outputs import TokenClassifierOutput

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha = None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
    def forward(self, logits, targets, mask):
        logits = logits.view(-1, logits.size(-1))
        targets = targets.view(-1)
        mask = mask.view(-1)

        logits = logits[mask]
        targets = targets[mask]

        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

class PhoBERT_Focal(nn.Module):
    def __init__(self, model_checkpoint, num_labels):
        super().__init__()
        self.num_labels = num_labels

        self.phobert = AutoModel.from_pretrained(model_checkpoint, add_pooling_layer=False)
        self.classifier = nn.Linear(self.phobert.config.hidden_size, num_labels)

        # Chỉ dùng Focal Loss, không có CRF
        self.focal_loss = FocalLoss(gamma=2.0)

    def forward(self, input_ids, attention_mask, labels=None, **kwargs):
        outputs = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state

        # Điểm số dự đoán thô
        logits = self.classifier(sequence_output)

        loss = None
        if labels is not None:
            # Mask bỏ qua các token -100
            loss_mask = (labels != -100)

            # Tính Focal Loss trực tiếp trên logits thay vì Cross Entropy
            loss = self.focal_loss(logits, labels, loss_mask)

        return TokenClassifierOutput(loss=loss, logits=logits)

In [ ]:
class PhoBERT_Focal_CRF(nn.Module):
    def __init__(self, model_checkpoint, num_labels, gamma=2.0, alpha=None):
        super().__init__()
        self.num_labels = num_labels
        self.phobert = AutoModel.from_pretrained(model_checkpoint, add_pooling_layer=False)
        self.classifier = nn.Linear(self.phobert.config.hidden_size, num_labels)

        # 1. Khởi tạo CRF layer
        self.crf = CRF(num_labels, batch_first=True)

        # 2. Khởi tạo Focal Loss
        self.focal_loss = FocalLoss(gamma=gamma, alpha=alpha)

    def forward(self, input_ids, attention_mask, labels=None, **kwargs):
        outputs = self.phobert(input_ids=input_ids, attention_mask=attention_mask)

        # Truyền thẳng đầu ra (không qua Dropout) vào bộ phân loại
        sequence_output = outputs.last_hidden_state
        logits = self.classifier(sequence_output)

        loss = None
        if labels is not None:
            # A. TÍNH CRF LOSS (Sequence-level)
            crf_mask = attention_mask.bool()
            safe_labels = labels.clone()
            safe_labels[safe_labels == -100] = 0 # Gán tạm nhãn 0 cho padding/sub_token
            crf_loss = -self.crf(logits, safe_labels, mask=crf_mask, reduction='mean')

            # B. TÍNH FOCAL LOSS (Token-level)
            focal_mask = (labels != -100) # Chỉ tính loss trên các token thật
            focal_loss_val = self.focal_loss(logits, labels, focal_mask)

            # C. TỔNG HỢP LOSS
            loss = crf_loss + focal_loss_val
        # D. GIẢI MÃ (DECODE) BẰNG CRF ĐỂ ĐÁNH GIÁ
        crf_mask_decode = attention_mask.bool()
        decoded_paths = self.crf.decode(logits, mask=crf_mask_decode)

        # Tạo fake_logits để tương thích hoàn toàn với hàm compute_metrics
        fake_logits = torch.zeros_like(logits)
        for i, path in enumerate(decoded_paths):
            for j, tag_id in enumerate(path):
                fake_logits[i, j, tag_id] = 1.0

        return TokenClassifierOutput(loss=loss, logits=fake_logits)

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

class ViMedNERDataset(Dataset):
    def __init__(self, sentences, tokenizer, label2id, max_length=128):
        self.sentences = sentences
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_length = max_length

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        sentence_data = self.sentences[idx]
        words = [item[0] for item in sentence_data]
        tags = [item[1] for item in sentence_data]

        # Tokenize từng từ và theo dõi số lượng sub-token của mỗi từ
        tokenized_outputs = []
        label_ids = []

        # Thêm token [CLS] đầu câu
        tokenized_outputs.append(self.tokenizer.cls_token_id)
        label_ids.append(-100)

        for word, tag in zip(words, tags):
            # Tokenize từng từ lẻ (không dùng is_split_into_words để tránh lỗi word_ids)
            sub_tokens = self.tokenizer.tokenize(word)
            sub_token_ids = self.tokenizer.convert_tokens_to_ids(sub_tokens)

            if len(sub_token_ids) > 0:
                # Sub-token đầu tiên nhận nhãn thật
                tokenized_outputs.append(sub_token_ids[0])
                label_ids.append(self.label2id[tag])

                # Các sub-token phía sau của cùng 1 từ nhận -100
                for sub_id in sub_token_ids[1:]:
                    tokenized_outputs.append(sub_id)
                    label_ids.append(-100)

        # Thêm token [SEP] cuối câu
        tokenized_outputs.append(self.tokenizer.sep_token_id)
        label_ids.append(-100)

        # Cắt ngắn (Truncation) nếu vượt quá max_length
        if len(tokenized_outputs) > self.max_length:
            tokenized_outputs = tokenized_outputs[:self.max_length]
            label_ids = label_ids[:self.max_length]

        # Tạo attention mask (1 cho token thật, 0 cho padding)
        attention_mask = [1] * len(tokenized_outputs)

        # Padding (Đệm) cho đủ max_length
        padding_length = self.max_length - len(tokenized_outputs)
        if padding_length > 0:
            tokenized_outputs = tokenized_outputs + [self.tokenizer.pad_token_id] * padding_length
            label_ids = label_ids + [-100] * padding_length
            attention_mask = attention_mask + [0] * padding_length

        item = {
            "input_ids": torch.tensor(tokenized_outputs, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(label_ids, dtype=torch.long)
        }
        return item

# 1. Khởi tạo Tokenizer
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

In [ ]:
import numpy as np
import evaluate
seqeval = evaluate.load("seqeval") # Khai báo seqeval ở đây
def compute_metrics(p):
    predictions, labels = p

    # Biến ma trận xác suất (logits) thành con số ID nhãn dự đoán cao nhất
    predictions = np.argmax(predictions, axis=2)

    # Lọc bỏ các nhãn -100 ra khỏi quá trình tính toán metric
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
import pickle
def load_dataset(file_path):
    with open(file_path,'rb') as f :
        dataset = pickle.load(f)
    return dataset

test_dataset_filepath = '/content/drive/MyDrive/datasetViMedNER/PreprocessedData/test_dataset.pkl'
test_dataset = load_dataset(test_dataset_filepath)

In [ ]:
import os
import torch
import pandas as pd
import safetensors.torch
from transformers import Trainer, TrainingArguments
# Đã thêm AutoTokenizer vào dòng import
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification, AutoTokenizer
from IPython.display import display

tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")
data_collator = DataCollatorForTokenClassification(tokenizer)

# 1. Khởi tạo danh sách lưu kết quả
results_table = []

# Cấu hình Trainer cơ bản chỉ dùng để đánh giá (Inference)
eval_args = TrainingArguments(
    output_dir="./eval_temp",
    per_device_eval_batch_size=16,
    report_to="none"
)

# 2. Hàm helper tự động nạp trọng số và lấy điểm
def evaluate_and_record(model_name, model_instance, checkpoint_path, dataset):
    print(f"⏳ Đang tải và đánh giá: {model_name}...")

    bin_path = os.path.join(checkpoint_path, "pytorch_model.bin")
    safetensor_path = os.path.join(checkpoint_path, "model.safetensors")

    # 1. Ưu tiên tuyệt đối nạp file pytorch_model.bin cho các mô hình Custom
    if os.path.exists(bin_path):
        state_dict = torch.load(bin_path, map_location="cpu")
        model_instance.load_state_dict(state_dict, strict=False)
        print(f"✅ Đã nạp thành công trọng số từ: pytorch_model.bin")

    # 2. Nếu không có file bin (ví dụ Baseline), mới dùng safetensors
    elif os.path.exists(safetensor_path):
        import safetensors.torch
        safetensors.torch.load_model(model_instance, safetensor_path)
        print(f"✅ Đã nạp thành công trọng số từ: model.safetensors")

    else:
        print(f"⚠️ Cảnh báo: Không tìm thấy file trọng số tại {checkpoint_path}")
        return

    model_instance.eval()

    # Tạo Trainer để chạy dự đoán
    eval_trainer = Trainer(
        model=model_instance,
        args=eval_args,
        eval_dataset=dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )
    metrics = eval_trainer.evaluate()
    results_table.append({
        "Mô hình (Model)": model_name,
        "Precision (%)": f"{metrics.get('eval_precision', 0) * 100:.2f}",
        "Recall (%)": f"{metrics.get('eval_recall', 0) * 100:.2f}",
        "F1-Score (%)": f"{metrics.get('eval_f1', 0) * 100:.2f}",
        "Accuracy (%)": f"{metrics.get('eval_accuracy', 0) * 100:.2f}"
    })

# 3. Khởi tạo 4 kiến trúc mô hình với trọng số rỗng
num_labels = len(label2id)

model_baseline = AutoModelForTokenClassification.from_pretrained("vinai/phobert-base-v2", num_labels=num_labels)
model_focal = PhoBERT_Focal("vinai/phobert-base-v2", num_labels=num_labels)
model_crf = PhoBERT_CRF("vinai/phobert-base-v2", num_labels=num_labels)
model_combined = PhoBERT_Focal_CRF("vinai/phobert-base-v2", num_labels=num_labels)

# 4. Chạy đánh giá trên tập test
# Đảm bảo biến test_dataset đã được load (ví dụ thông qua pickle.load) ở các ô code trước
target_dataset = test_dataset

evaluate_and_record("1. Baseline (CE Loss)", model_baseline, "/content/drive/MyDrive/vimedner_final_baseline", target_dataset)
evaluate_and_record("2. PhoBERT + Focal Loss", model_focal, "/content/drive/MyDrive/vimedner_final_focal", target_dataset)
evaluate_and_record("3. PhoBERT + CRF", model_crf, "/content/drive/MyDrive/vimedner_final_crf", target_dataset)
evaluate_and_record("4. ALESR-ViMedNER", model_combined, "/content/drive/MyDrive/phobert_focal_crf", target_dataset)

# 5. Xuất bảng tổng hợp
df_compare = pd.DataFrame(results_table)

print("\n" + "="*70)
print("🏆 BẢNG TỔNG HỢP SO SÁNH HIỆU NĂNG 4 MÔ HÌNH VIMEDNER")
print("="*70)
display(df_compare)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForTokenClassification LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
classifier.bias           | MISSING    | 
classifier.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Training Loss,Validation Loss,Step,Precision,Recall,F1,Accuracy
No log,13.944240,0,0.702682,0.719733,0.711105,0.898380


⏳ Đang tải và đánh giá: 4. ALESR-ViMedNER...
✅ Đã nạp thành công trọng số từ: pytorch_model.bin


Training Loss,Validation Loss,Step,Precision,Recall,F1,Accuracy
No log,7.061285,0,0.732718,0.698133,0.715008,0.897369



🏆 BẢNG TỔNG HỢP SO SÁNH HIỆU NĂNG 4 MÔ HÌNH VIMEDNER


,Mô hình (Model),Precision (%),Recall (%),F1-Score (%),Accuracy (%)
0,1. Baseline (CE Loss),60.41,70.72,65.16,89.40
1,2. PhoBERT + Focal Loss,66.95,72.40,69.57,90.03
2,3. PhoBERT + CRF,70.27,71.97,71.11,89.84
3,4. ALESR-ViMedNER,73.27,69.81,71.50,89.74


In [ ]:
import os
import torch
import pandas as pd
import safetensors.torch
from transformers import Trainer, TrainingArguments
# Đã thêm AutoTokenizer vào dòng import
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification, AutoTokenizer
from IPython.display import display

dev_dataset_filepath = '/content/drive/MyDrive/datasetViMedNER/PreprocessedData/dev_dataset.pkl'
dev_dataset = load_dataset(dev_dataset_filepath)
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")
data_collator = DataCollatorForTokenClassification(tokenizer)

# 1. Khởi tạo danh sách lưu kết quả
results_table = []

# Cấu hình Trainer cơ bản chỉ dùng để đánh giá (Inference)
eval_args = TrainingArguments(
    output_dir="./eval_temp",
    per_device_eval_batch_size=16,
    report_to="none"
)

# 2. Hàm helper tự động nạp trọng số và lấy điểm
def evaluate_and_record(model_name, model_instance, checkpoint_path, dataset):
    print(f"⏳ Đang tải và đánh giá: {model_name}...")

    safetensor_path = os.path.join(checkpoint_path, "model.safetensors")
    bin_path = os.path.join(checkpoint_path, "pytorch_model.bin")

    if os.path.exists(safetensor_path):
        safetensors.torch.load_model(model_instance, safetensor_path)
    elif os.path.exists(bin_path):
        model_instance.load_state_dict(torch.load(bin_path))
    else:
        print(f"⚠️ Cảnh báo: Không tìm thấy file trọng số tại {checkpoint_path}")
        return

    # Tạo Trainer để chạy dự đoán
    eval_trainer = Trainer(
        model=model_instance,
        args=eval_args,
        eval_dataset=dataset, # ĐÃ SỬA: Dùng tham số 'dataset' truyền vào hàm
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    metrics = eval_trainer.evaluate()

    results_table.append({
        "Mô hình (Model)": model_name,
        "Precision (%)": f"{metrics.get('eval_precision', 0) * 100:.2f}",
        "Recall (%)": f"{metrics.get('eval_recall', 0) * 100:.2f}",
        "F1-Score (%)": f"{metrics.get('eval_f1', 0) * 100:.2f}",
        "Accuracy (%)": f"{metrics.get('eval_accuracy', 0) * 100:.2f}"
    })

# 3. Khởi tạo 4 kiến trúc mô hình với trọng số rỗng
num_labels = len(label2id)

model_baseline = AutoModelForTokenClassification.from_pretrained("vinai/phobert-base-v2", num_labels=num_labels)
model_focal = PhoBERT_Focal("vinai/phobert-base-v2", num_labels=num_labels)
model_crf = PhoBERT_CRF("vinai/phobert-base-v2", num_labels=num_labels)
model_combined = PhoBERT_Focal_CRF("vinai/phobert-base-v2", num_labels=num_labels)

# 4. Chạy đánh giá trên tập test
# Đảm bảo biến test_dataset đã được load (ví dụ thông qua pickle.load) ở các ô code trước
target_dataset = test_dataset

evaluate_and_record("1. Baseline (CE Loss)", model_baseline, "/content/drive/MyDrive/vimedner_final_baseline", dev_dataset)
evaluate_and_record("2. PhoBERT + Focal Loss", model_focal, "/content/drive/MyDrive/vimedner_final_focal", dev_dataset)
evaluate_and_record("3. PhoBERT + CRF", model_crf, "/content/drive/MyDrive/vimedner_final_crf", dev_dataset)
evaluate_and_record("4. ALESR-ViMedNER", model_combined, "/content/drive/MyDrive/phobert_focal_crf", dev_dataset)

# 5. Xuất bảng tổng hợp
df_compare = pd.DataFrame(results_table)

print("\n" + "="*70)
print("🏆 BẢNG TỔNG HỢP SO SÁNH HIỆU NĂNG 4 MÔ HÌNH VIMEDNER")
print("="*70)
display(df_compare)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForTokenClassification LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
classifier.bias           | MISSING    | 
classifier.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Training Loss,Validation Loss,Step,Precision,Recall,F1,Accuracy
No log,0.287524,0,0.641381,0.758121,0.694882,0.919809


⏳ Đang tải và đánh giá: 2. PhoBERT + Focal Loss...
⚠️ Cảnh báo: Không tìm thấy file trọng số tại /content/drive/MyDrive/vimedner_final_focal
⏳ Đang tải và đánh giá: 3. PhoBERT + CRF...
⚠️ Cảnh báo: Không tìm thấy file trọng số tại /content/drive/MyDrive/vimedner_final_crf
⏳ Đang tải và đánh giá: 4. ALESR-ViMedNER...


Training Loss,Validation Loss,Step,Precision,Recall,F1,Accuracy
No log,7.015018,0,0.730534,0.705235,0.717662,0.900385



🏆 BẢNG TỔNG HỢP SO SÁNH HIỆU NĂNG 4 MÔ HÌNH VIMEDNER


,Mô hình (Model),Precision (%),Recall (%),F1-Score (%),Accuracy (%)
0,1. Baseline (CE Loss),64.14,75.81,69.49,91.98
1,4. ALESR-ViMedNER,73.05,70.52,71.77,90.04
